In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# ตั้งค่าระดับการปรับแต่งของ Transpiler

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
อุปกรณ์ควอนตัมจริงนั้นมีสัญญาณรบกวนและข้อผิดพลาดของ Gate ดังนั้นการปรับแต่ง Circuit เพื่อลดความลึกและจำนวน Gate จึงสามารถปรับปรุงผลลัพธ์ที่ได้จากการรัน Circuit เหล่านั้นได้อย่างมาก
ฟังก์ชัน [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) มีอาร์กิวเมนต์ตำแหน่งบังคับหนึ่งตัวคือ `optimization_level` ซึ่งควบคุมความพยายามที่ Transpiler ใช้ในการปรับแต่ง Circuit อาร์กิวเมนต์นี้เป็นจำนวนเต็มที่รับค่าได้คือ 0, 1, 2 หรือ 3
ระดับการปรับแต่งที่สูงขึ้นจะสร้าง Circuit ที่ปรับแต่งมากขึ้นแต่ใช้เวลา compile นานขึ้น
ตารางต่อไปนี้อธิบายการปรับแต่งที่ทำในแต่ละการตั้งค่า

<table>
  <thead>
    <tr>
      <th>ระดับการปรับแต่ง</th>
      <th>คำอธิบาย</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        ไม่มีการปรับแต่ง: โดยทั่วไปใช้สำหรับการกำหนดคุณสมบัติของฮาร์ดแวร์
        - การแปลพื้นฐาน
        - Layout/Routing: `TrivialLayout` ซึ่งเลือกหมายเลข Qubit ทางกายภาพเดียวกับ Qubit เสมือน และแทรก SWAP เพื่อให้ทำงานได้ (โดยใช้ `SabreSwap`)
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        การปรับแต่งระดับเบา:
        -   Layout/Routing: ลอง Layout ด้วย `TrivialLayout` ก่อน ถ้าต้องการ SWAP เพิ่มเติม จะหา Layout ที่มีจำนวน SWAP น้อยที่สุดโดยใช้ `SabreSwap` แล้วใช้ `VF2LayoutPostLayout` เพื่อเลือก Qubit ที่ดีที่สุดในกราฟ
        -   `InverseCancellation`
        -   การปรับแต่ง Gate แบบ 1Q
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        การปรับแต่งระดับกลาง:
          - Layout/Routing: ระดับการปรับแต่ง 1 (ไม่มี trivial) + heuristic ที่ปรับแต่งด้วยความลึกการค้นหาและจำนวนการทดลองฟังก์ชันการปรับแต่งที่มากขึ้น เนื่องจากไม่ใช้ `TrivialLayout` จึงไม่มีการพยายามใช้หมายเลข Qubit ทางกายภาพและเสมือนเหมือนกัน
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        การปรับแต่งระดับสูง:
        - ระดับการปรับแต่ง 2 + heuristic ที่ปรับแต่งใน layout/routing มากขึ้นด้วยความพยายาม/การทดลองมากขึ้น
        - การสังเคราะห์ใหม่ของบล็อก Gate สองตัวโดยใช้ [Cartan's KAK Decomposition](https://arxiv.org/abs/quant-ph/0507171)
        - Passes ที่ทำลาย unitarity:
          * `OptimizeSwapBeforeMeasure`: ย้ายการวัดเพื่อหลีกเลี่ยง SWAP
          * `RemoveDiagonalGatesBeforeMeasure`: ลบ Gate ก่อนการวัดที่ไม่มีผลต่อการวัด
      </td>
    </tr>
  </tbody>
</table>

## ระดับการปรับแต่งในทางปฏิบัติ
เนื่องจาก Gate สองตัวโดยทั่วไปเป็นแหล่งข้อผิดพลาดที่สำคัญที่สุด เราจึงสามารถประมาณ "ประสิทธิภาพฮาร์ดแวร์" ของการ transpile ได้โดยการนับจำนวน Gate สองตัวใน Circuit ที่ได้
ที่นี่ เราจะลองระดับการปรับแต่งต่างๆ กับ Circuit อินพุตที่ประกอบด้วย unitary สุ่มตามด้วย SWAP Gate

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

เราจะใช้ mock Backend `FakeSherbrooke` ในตัวอย่าง ก่อนอื่น มา transpile โดยใช้ระดับการปรับแต่ง 0

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

Circuit ที่ transpile แล้วมี ECR Gate สองตัวจำนวนหก Gate

ทำซ้ำสำหรับระดับการปรับแต่ง 1:

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

Circuit ที่ transpile แล้วยังคงมี ECR Gate หกตัว แต่จำนวน Gate แบบ single-qubit ลดลง

ทำซ้ำสำหรับระดับการปรับแต่ง 2:

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

ผลลัพธ์เหมือนกับระดับการปรับแต่ง 1 โปรดทราบว่าการเพิ่มระดับการปรับแต่งไม่ได้ให้ผลเสมอไป

ทำซ้ำอีกครั้ง สำหรับระดับการปรับแต่ง 3:

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

ตอนนี้มีเพียง ECR Gate สามตัว เราได้ผลลัพธ์นี้เพราะที่ระดับการปรับแต่ง 3 Qiskit พยายามสังเคราะห์บล็อก Gate สองตัวใหม่ และ Gate สองตัวใดๆ สามารถนำมาใช้งานโดยใช้ ECR Gate ไม่เกินสาม Gate เราสามารถได้ ECR Gate น้อยลงอีกถ้าตั้งค่า `approximation_degree` ให้น้อยกว่า 1 ซึ่งยอมให้ Transpiler ทำการประมาณค่าที่อาจนำข้อผิดพลาดบางส่วนมาใน Gate decomposition (ดู [พารามิเตอร์ที่ใช้บ่อยสำหรับการ transpilation](common-parameters#approximation-degree)):

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

Circuit นี้มีเพียง ECR Gate สองตัว แต่เป็น Circuit แบบประมาณ เพื่อทำความเข้าใจว่าผลกระทบของมันแตกต่างจาก Circuit แบบแม่นยำอย่างไร เราสามารถคำนวณ fidelity ระหว่าง unitary operator ที่ Circuit นี้ implement และ unitary แบบแม่นยำ ก่อนทำการคำนวณ เราจะลด Circuit ที่ transpile แล้วซึ่งมี 127 Qubit ลงมาเป็น Circuit ที่มีเฉพาะ Qubit ที่ active ซึ่งมีอยู่สองตัว

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>